In [2]:
# ============================================================
# INSTALL LIBRARIES
# ============================================================

!pip install -q -U transformers datasets accelerate scikit-learn pandas tqdm

# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# WRITE FILE
# ============================================================



import os
import gc
import json
import glob
import time
import shutil
import random
import logging
from collections import OrderedDict

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader, Subset

from datasets import load_dataset

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

# ============================================================
# CONFIG
# ============================================================

class Config:

    model_name = "huawei-noah/TinyBERT_General_4L_312D"

    dataset_name = "imdb"

    setting = "F-TinyBERT-FFT"

    num_labels = 2

    # ========================================================
    # TRAINING
    # ========================================================

    batch_size = 4

    learning_rate = 2e-5

    rounds = 10

    local_epochs = 1

    gradient_accumulation_steps = 4

    fp16 = True

    max_length = 256

    weight_decay = 0.01

    warmup_ratio = 0.1

    max_grad_norm = 1.0

    # ========================================================
    # FEDERATED
    # ========================================================

    num_clients = 5

    partition_type = "noniid"

    alpha = 0.5

    # ========================================================
    # EARLY STOPPING
    # ========================================================

    patience = 3

    # ========================================================
    # DATA
    # ========================================================

    validation_ratio = 0.1

    num_workers = 0

    # ========================================================
    # CHECKPOINT
    # ========================================================

    keep_last_k = 2

    resume = True

    # ========================================================
    # SYSTEM
    # ========================================================

    seed = 42

    output_dir = (
        "/content/drive/MyDrive/"
        "federated_tinybert_imdb"
    )

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )


cfg = Config()

# ============================================================
# OUTPUT PATHS
# ============================================================

CHECKPOINT_DIR = os.path.join(
    cfg.output_dir,
    "checkpoints"
)

BEST_MODEL_DIR = os.path.join(
    cfg.output_dir,
    "best_model"
)

FINAL_MODEL_DIR = os.path.join(
    cfg.output_dir,
    "final_model"
)

HISTORY_JSON = os.path.join(
    cfg.output_dir,
    "history.json"
)

HISTORY_CSV = os.path.join(
    cfg.output_dir,
    "history.csv"
)

TRAIN_LOG = os.path.join(
    cfg.output_dir,
    "train.log"
)

CONFIG_JSON = os.path.join(
    cfg.output_dir,
    "config.json"
)

# ============================================================
# CREATE DIRS
# ============================================================

os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# ============================================================
# LOGGER
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(TRAIN_LOG),
        logging.StreamHandler(),
    ],
)

logger = logging.getLogger(__name__)

# ============================================================
# SAVE CONFIG
# ============================================================

with open(CONFIG_JSON, "w") as f:
    json.dump(cfg.__dict__, f, indent=4)

# ============================================================
# SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)


set_seed(cfg.seed)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    cfg.model_name
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

# ============================================================
# LOAD DATASET
# ============================================================

dataset = load_dataset("imdb")

split_dataset = dataset["train"].train_test_split(
    test_size=cfg.validation_ratio,
    seed=cfg.seed,
)

train_dataset = split_dataset["train"]

valid_dataset = split_dataset["test"]

# ============================================================
# TOKENIZATION
# ============================================================

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=cfg.max_length,
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True,
)

train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

valid_dataset = valid_dataset.rename_column(
    "label",
    "labels"
)

train_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

valid_dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels",
    ],
)

# ============================================================
# IID PARTITION
# ============================================================

def iid_partition(dataset, num_clients):

    indices = np.random.permutation(
        len(dataset)
    )

    splits = np.array_split(
        indices,
        num_clients
    )

    return [s.tolist() for s in splits]

# ============================================================
# NON-IID PARTITION
# ============================================================

def dirichlet_partition(
    dataset,
    num_clients,
    alpha
):

    labels = np.array(dataset["labels"])

    classes = np.unique(labels)

    client_indices = [[] for _ in range(num_clients)]

    for c in classes:

        idx = np.where(labels == c)[0]

        np.random.shuffle(idx)

        proportions = np.random.dirichlet(
            [alpha] * num_clients
        )

        proportions = (
            np.cumsum(proportions)
            * len(idx)
        ).astype(int)[:-1]

        split_idx = np.split(idx, proportions)

        for client_id, part in enumerate(split_idx):

            client_indices[client_id].extend(
                part.tolist()
            )

    return client_indices

# ============================================================
# CREATE PARTITIONS
# ============================================================

if cfg.partition_type == "iid":

    client_partitions = iid_partition(
        train_dataset,
        cfg.num_clients,
    )

else:

    client_partitions = dirichlet_partition(
        train_dataset,
        cfg.num_clients,
        cfg.alpha,
    )

# ============================================================
# DATALOADER
# ============================================================

client_loaders = []

client_num_samples = []

for client_id in range(cfg.num_clients):

    subset = Subset(
        train_dataset,
        client_partitions[client_id]
    )

    loader = DataLoader(
        subset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collator,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    client_loaders.append(loader)

    client_num_samples.append(len(subset))

valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collator,
)

# ============================================================
# CREATE MODEL
# ============================================================

def create_model():

    model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            cfg.model_name,
            num_labels=cfg.num_labels,
        )
    )

    return model

# ============================================================
# PARAM COUNT
# ============================================================

def count_parameters(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total, trainable

# ============================================================
# EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    losses = []

    preds_all = []

    labels_all = []

    for batch in tqdm(loader, leave=False):

        batch = {
            k: v.to(cfg.device)
            for k, v in batch.items()
        }

        with torch.cuda.amp.autocast(
            enabled=cfg.fp16
        ):

            outputs = model(**batch)

            loss = outputs.loss

            logits = outputs.logits

        preds = torch.argmax(
            logits,
            dim=-1
        )

        losses.append(loss.item())

        preds_all.extend(
            preds.cpu().numpy()
        )

        labels_all.extend(
            batch["labels"].cpu().numpy()
        )

    accuracy = accuracy_score(
        labels_all,
        preds_all
    )

    precision, recall, macro_f1, _ = (
        precision_recall_fscore_support(
            labels_all,
            preds_all,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "eval_loss": float(np.mean(losses)),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "macro_f1": float(macro_f1),
    }

# ============================================================
# FEDAVG
# ============================================================

def fedavg(states, weights):

    avg_state = OrderedDict()

    total_weight = sum(weights)

    for key in states[0].keys():

        avg_state[key] = sum(
            state[key] * (w / total_weight)
            for state, w in zip(states, weights)
        )

    return avg_state

# ============================================================
# CLIENT TRAINING
# ============================================================

def train_client(global_state, client_loader):

    model = create_model()

    model.load_state_dict(global_state)

    model.to(cfg.device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
    )

    total_steps = (
        len(client_loader)
        * cfg.local_epochs
    ) // cfg.gradient_accumulation_steps

    warmup_steps = int(
        cfg.warmup_ratio * total_steps
    )

    scheduler = (
        get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=cfg.fp16
    )

    model.train()

    losses = []

    start_time = time.time()

    optimizer.zero_grad()

    for epoch in range(cfg.local_epochs):

        pbar = tqdm(client_loader, leave=False)

        for step, batch in enumerate(pbar):

            batch = {
                k: v.to(cfg.device)
                for k, v in batch.items()
            }

            with torch.cuda.amp.autocast(
                enabled=cfg.fp16
            ):

                outputs = model(**batch)

                loss = outputs.loss

                loss = (
                    loss
                    / cfg.gradient_accumulation_steps
                )

            scaler.scale(loss).backward()

            if (
                (step + 1)
                % cfg.gradient_accumulation_steps
                == 0
            ):

                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    cfg.max_grad_norm,
                )

                scaler.step(optimizer)

                scaler.update()

                scheduler.step()

                optimizer.zero_grad()

            losses.append(loss.item())

    train_time = time.time() - start_time

    state = {
        k: v.cpu()
        for k, v in model.state_dict().items()
    }

    del model

    gc.collect()

    torch.cuda.empty_cache()

    return (
        state,
        float(np.mean(losses)),
        float(train_time),
    )

# ============================================================
# MAIN
# ============================================================

def main():

    logger.info("=" * 80)

    logger.info("F-TinyBERT-FFT IMDB")

    logger.info(f"Device: {cfg.device}")

    logger.info("=" * 80)

    global_model = create_model()

    global_model.to(cfg.device)

    best_metric = -1

    patience_counter = 0

    history = []

    total_params, trainable_params = (
        count_parameters(global_model)
    )

    for round_idx in range(
        1,
        cfg.rounds + 1,
    ):

        logger.info(f"ROUND {round_idx}")

        round_start = time.time()

        global_state = {
            k: v.cpu()
            for k, v in global_model.state_dict().items()
        }

        client_states = []

        client_losses = []

        client_times = []

        for client_id in range(cfg.num_clients):

            logger.info(
                f"Training Client {client_id}"
            )

            (
                client_state,
                client_loss,
                client_time,
            ) = train_client(
                global_state,
                client_loaders[client_id],
            )

            client_states.append(client_state)

            client_losses.append(client_loss)

            client_times.append(client_time)

        aggregated_state = fedavg(
            client_states,
            client_num_samples,
        )

        global_model.load_state_dict(
            aggregated_state
        )

        metrics = evaluate(
            global_model,
            valid_loader,
        )

        round_time = (
            time.time() - round_start
        )

        is_new_best = (
            metrics["macro_f1"]
            > best_metric
        )

        if is_new_best:

            best_metric = metrics["macro_f1"]

            patience_counter = 0

            global_model.save_pretrained(
                BEST_MODEL_DIR
            )

            tokenizer.save_pretrained(
                BEST_MODEL_DIR
            )

        else:

            patience_counter += 1

        row = {

            "round": round_idx,

            "train_loss":
                float(np.mean(client_losses)),

            "eval_loss":
                float(metrics["eval_loss"]),

            "accuracy":
                float(metrics["accuracy"]),

            "precision":
                float(metrics["precision"]),

            "recall":
                float(metrics["recall"]),

            "macro_f1":
                float(metrics["macro_f1"]),

            "round_time":
                float(round_time),

            "client_train_time":
                float(np.sum(client_times)),

            "trainable_params":
                int(trainable_params),

            "total_params":
                int(total_params),

            "model_name":
                cfg.model_name,

            "dataset_name":
                cfg.dataset_name,

            "setting":
                cfg.setting,

            "num_clients":
                cfg.num_clients,

            "local_epochs":
                cfg.local_epochs,

            "partition_type":
                cfg.partition_type,

            "best_metric_so_far":
                float(best_metric),

            "patience_counter":
                int(patience_counter),

            "is_new_best":
                bool(is_new_best),
        }

        history.append(row)

        pd.DataFrame(history).to_csv(
            HISTORY_CSV,
            index=False,
        )

        with open(HISTORY_JSON, "w") as f:

            json.dump(
                history,
                f,
                indent=4,
            )

        logger.info(
            json.dumps(
                row,
                indent=4,
            )
        )

        if patience_counter >= cfg.patience:

            logger.info("EARLY STOPPING")

            break

    global_model.save_pretrained(
        FINAL_MODEL_DIR
    )

    tokenizer.save_pretrained(
        FINAL_MODEL_DIR
    )

    logger.info("TRAINING FINISHED")

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

# ============================================================
# RUN TRAINING
# ============================================================



Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

  0%|          | 0/127 [00:00<?, ?it/s]

  0%|          | 0/79 [00:00<?, ?it/s]

/tmp/ipykernel_2233/2674186053.py:475: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]